# 🇺🇸 获取美股行情数据

---

### 🎯 目标
- 用 `yfinance` 下载美股历史数据
- 批量对比多只股票走势
- 查看公司基本面

### 📌 yfinance 特点
- 完全免费，不需要注册
- 数据来自 Yahoo Finance
- 支持股票、ETF、指数、加密货币

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC']
matplotlib.rcParams['axes.unicode_minus'] = False
print("✅ 库导入成功")

## 1️⃣ 下载单只股票数据

In [ ]:
# 下载苹果 6 个月日线
aapl = yf.download("AAPL", period="6mo")
print(f"✅ 获取 {len(aapl)} 天 AAPL 数据")
aapl.tail()

In [ ]:
# 价格走势 + 成交量
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(aapl.index, aapl["Close"], color="steelblue", linewidth=1.5)
ma20 = aapl["Close"].rolling(20).mean()
ma50 = aapl["Close"].rolling(50).mean()
ax1.plot(aapl.index, ma20, label="MA20", color="orange", linewidth=1, linestyle="--")
ax1.plot(aapl.index, ma50, label="MA50", color="red", linewidth=1, linestyle="--")
ax1.set_title("AAPL 走势图", fontsize=14)
ax1.set_ylabel("价格 ($)")
ax1.legend()
ax1.grid(True, alpha=0.3)

colors = ["green" if aapl["Close"].iloc[i] >= aapl["Close"].iloc[max(0,i-1)] else "red" for i in range(len(aapl))]
ax2.bar(aapl.index, aapl["Volume"], color=colors, alpha=0.7, width=1)
ax2.set_ylabel("成交量")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2️⃣ 多股票对比 — 谁跑得最快？

In [ ]:
tickers = ["AAPL", "GOOGL", "MSFT", "NVDA", "TSLA"]
data = yf.download(tickers, period="6mo")
closes = data["Close"]

# 归一化（起始=100）方便对比
normalized = closes / closes.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 7))
for t in tickers:
    ax.plot(normalized.index, normalized[t], linewidth=1.5, label=t)

ax.axhline(y=100, color="gray", linestyle="--", linewidth=0.5)
ax.set_title("科技巨头走势对比（归一化 = 100）", fontsize=14)
ax.set_ylabel("相对价格")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 排名
total_ret = (closes.iloc[-1] / closes.iloc[0] - 1) * 100
print("📊 6个月收益率排名")
for i, (t, r) in enumerate(total_ret.sort_values(ascending=False).items(), 1):
    medal = ["🥇", "🥈", "🥉", "  ", "  "][i-1]
    bar = "█" * max(0, int(r / 2))
    print(f"  {medal} {t:<5} {r:>+7.2f}%  {bar}")

## 3️⃣ 公司基本面

In [ ]:
stock = yf.Ticker("AAPL")
info = stock.info

print("┌────────────────────────────────────────┐")
print("│          🍎 AAPL 基本面信息             │")
print("├────────────────────────────────────────┤")
print(f"│ 公司: {info.get('longName', 'N/A'):<32}│")
print(f"│ 行业: {info.get('industry', 'N/A'):<32}│")
print(f"│ 市值: ${info.get('marketCap',0)/1e12:.2f}T{'':<26}│")
print(f"│ PE:   {info.get('trailingPE', 'N/A'):<33}│")
print(f"│ 股息: {info.get('dividendYield',0)*100:.2f}%{'':<29}│")
print(f"│ 52周高: ${info.get('fiftyTwoWeekHigh', 0):>8.2f}{'':<22}│")
print(f"│ 52周低: ${info.get('fiftyTwoWeekLow', 0):>8.2f}{'':<22}│")
print("└────────────────────────────────────────┘")

## 4️⃣ 收益率分析

In [ ]:
returns = aapl["Close"].pct_change().dropna() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 收益率分布
axes[0].hist(returns, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(x=0, color="red", linestyle="--")
axes[0].axvline(x=returns.mean(), color="orange", linestyle="--", label=f"均值: {returns.mean():.3f}%")
axes[0].set_title("AAPL 日收益率分布")
axes[0].set_xlabel("日收益率 (%)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 累计收益率
cum_ret = (1 + aapl["Close"].pct_change()).cumprod() - 1
axes[1].plot(cum_ret.index, cum_ret * 100, color="steelblue", linewidth=1.5)
axes[1].fill_between(cum_ret.index, cum_ret * 100, 0, alpha=0.1, color="steelblue")
axes[1].axhline(y=0, color="gray", linestyle="-", linewidth=0.5)
axes[1].set_title("AAPL 累计收益率")
axes[1].set_ylabel("累计收益率 (%)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📝 常用代码参考

| 类别 | 代码 | 名称 |
|------|------|------|
| 科技 | AAPL, GOOGL, MSFT, NVDA, META, TSLA | 苹果/谷歌/微软/英伟达/Meta/特斯拉 |
| ETF | SPY, QQQ, DIA, IWM | 标普500/纳指100/道琼斯/罗素2000 |
| 加密 | COIN, MSTR, BITO | Coinbase/MicroStrategy/比特币ETF |

---

**下一个** → `03_K线图可视化.ipynb`